In [1]:
%env PYTHONUNBUFFERED=1

import os
from pathlib import Path



if Path.cwd().name == 'notebooks':
    os.chdir('..')
    print(f'changed working directory to: {Path.cwd()}')

env: PYTHONUNBUFFERED=1
changed working directory to: c:\Users\neucl\Dev\Thesis\cxr-diff-vqa


In [2]:
from PIL import Image
from lib.utils import setup_logging

log_file ="logs/resize.log"
logger = setup_logging(log_file=log_file)


def resize_image(args):
    """
    Worker function to resize a single image.
    Creates parent directories if they don't exist.
    """
    source_path_str, dest_path_str, size = args
    source_path = Path(source_path_str)
    dest_path = Path(dest_path_str)

    # logger.debug(f"Worker processing: {source_path}")
    
    try:
        # ensure the destination directory exists
        dest_path.parent.mkdir(parents=True, exist_ok=True)

        # open, resize, and save the image
        with Image.open(source_path) as img:

            # using LANCZOS for high-quality downsampling
            img_resized = img.resize((size, size), Image.Resampling.LANCZOS)

            # convert to RGB if it's grayscale to ensure consistency
            if img_resized.mode != 'RGB':
                img_resized = img_resized.convert('RGB')
            img_resized.save(dest_path, 'JPEG', quality=95)

        return None
    except Exception as e:
        logger.error(f"Error processing {source_path}: {e}")
        return f"Error processing {source_path}: {e}"

In [3]:
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

def run_resizing(source_dir_str, dest_dir_str, target_size=224, num_workers=1):
    """
    Main logic adapted for notebook execution, using imap_unordered and string paths.
    Includes a sequential path for debugging (num_workers=1).
    """
    source_dir = Path(source_dir_str)
    dest_dir = Path(dest_dir_str)
    size = target_size

    logger.info(f"Source directory: {source_dir}")
    logger.info(f"Destination directory: {dest_dir}")
    logger.info(f"Target size: {size}x{size}")

    # 1. Finding files (unchanged)
    logger.info("Finding all JPG files...")
    source_files_generator = source_dir.rglob('*.jpg')
    logger.info("Counting files...")
    temp_source_files_list = list(source_files_generator)
    total_files = len(temp_source_files_list)
    logger.info(f"Found {total_files} images to resize.")
    if not total_files:
        logger.error("Error: No .jpg files found.")
        return

    # 2. Task generator (unchanged)
    def task_generator():
        for source_path in temp_source_files_list:
             relative_path = source_path.relative_to(source_dir)
             dest_path = dest_dir / relative_path
             yield (str(source_path), str(dest_path), size)

    # 3. --- MODIFIED: Use sequential processing if num_workers is 1 ---
    logger.info("Starting image resizing...")
    # --- SET num_workers TO 1 FOR DEBUGGING ---
    # num_workers = 1
    # ------------------------------------------
    logger.info(f"Using {num_workers} worker process(es).")

    errors_list = []
    processed_count = 0

    # if num_workers > 1:
    #     # Use ProcessPoolExecutor for parallel processing
    #     with ProcessPoolExecutor(max_workers=num_workers) as executor:
    #         results_iterator = executor.map(resize_image, task_generator())
    #         with tqdm(total=total_files, desc="Resizing Images (Parallel)") as pbar:
    #             for result in results_iterator:
    #                 if result is not None:
    #                     errors_list.append(result)
    #                 processed_count += 1
    #                 pbar.update(1)
    # else:
    #     # Run sequentially for debugging
    #     logger.info("running sequentially")
    #     tasks = list(task_generator()) # Need the full list for sequential iteration
    #     with tqdm(total=total_files, desc="Resizing Images (Sequential)") as pbar:
    #         for task_args in tasks:
    #             # Call the resize function directly
    #             result = resize_image(task_args)
    #             if result is not None:
    #                 errors_list.append(result)
    #             processed_count += 1
    #             pbar.update(1)

    logger.info("running sequentially")
    tasks = list(task_generator()) # Need the full list for sequential iteration
    with tqdm(total=total_files, desc="Resizing Images (Sequential)") as pbar:
        for task_args in tasks:
            # Call the resize function directly
            result = resize_image(task_args)
            if result is not None:
                errors_list.append(result)
            processed_count += 1
            pbar.update(1)

    logger.info(f"Processed {processed_count} tasks.")

    if errors_list:
        logger.warning("\n--- Errors occurred during processing ---")
        for error in errors_list[:20]:
             logger.warning(error)
        if len(errors_list) > 20:
             logger.warning(f"... and {len(errors_list) - 20} more errors.")

    logger.info("\nResizing complete!")
    logger.info(f"Resized images are saved in: {dest_dir}")

### images left to process

- D:\MIMIC-CXR-JPG\p15\p15614836\s55027798\f944693e-1cb4440b-d13fa51b-a02f27b3-f97efdff.jpg

In [4]:
source_directory = "D:/MIMIC-CXR-JPG/"      
destination_directory = "E:/MIMIC-CXR-JPG-224x224/" 
image_size = 224

run_resizing(source_directory, destination_directory, image_size)

2025-10-29 17:44:53,366 - root - INFO - Source directory: D:\MIMIC-CXR-JPG
2025-10-29 17:44:53,366 - root - INFO - Destination directory: E:\MIMIC-CXR-JPG-224x224
2025-10-29 17:44:53,366 - root - INFO - Target size: 224x224
2025-10-29 17:44:53,374 - root - INFO - Finding all JPG files...
2025-10-29 17:44:53,374 - root - INFO - Counting files...
2025-10-29 17:50:38,610 - root - INFO - Found 377110 images to resize.
2025-10-29 17:50:38,611 - root - INFO - Starting image resizing...
2025-10-29 17:50:38,611 - root - INFO - Using 1 worker process(es).
2025-10-29 17:50:38,611 - root - INFO - running sequentially
Resizing Images (Sequential):   4%|▎         | 13988/377110 [15:47<7:28:17, 13.50it/s] 2025-10-29 18:06:41,498 - PIL.TiffImagePlugin - DEBUG - tag: Orientation (274) - type: short (3) - value: b'\x00\x01'
2025-10-29 18:06:41,498 - PIL.TiffImagePlugin - DEBUG - tag: ExifIFD (34665) - type: long (4) - value: b'\x00\x00\x00&'
Resizing Images (Sequential):  11%|█         | 41478/377110 [

# Fixing the failed image

In [2]:
from PIL import Image

source_path = 'D:\\MIMIC-CXR-JPG\\p15\\p15614836\\s55027798\\f944693e-1cb4440b-d13fa51b-a02f27b3-f97efdff.jpg'
dest_path = 'E:\\MIMIC-CXR-JPG-224x224\\p15\\p15614836\\s55027798\\f944693e-1cb4440b-d13fa51b-a02f27b3-f97efdff.jpg'
size = 224

with Image.open(source_path) as img:

    img_resized = img.resize((size, size), Image.Resampling.LANCZOS)

    if img_resized.mode != 'RGB':
        img_resized = img_resized.convert('RGB')
    img_resized.save(dest_path, 'JPEG', quality=95)
